# 00 Setup and Data Dictionary

This notebook defines the project-level path conventions, source hierarchy, core variable names, forecast notation, and methodological guardrails used by the forecasting pipeline. It also provides lightweight file-availability checks.

The notebook does not construct datasets, estimate models, execute legacy notebooks, or change methodology.


## Pipeline position

This is the setup and navigation notebook for the active forecasting pipeline. It precedes the data parsing, panel construction, modelling, and thesis-output notebooks. Its role is to document shared conventions and verify that expected project files can be located without running heavy processing.


## Inputs

The notebook relies on the project instruction files that define working rules, inventory status, replication planning, and project navigation. It also checks the current transaction, weather, and master-panel files used by the existing pipeline checkpoint. Optional or historical data files are inspected only if present.


## Outputs

This notebook writes no Parquet, CSV, HTML, or PNG outputs. When executed, it displays path and metadata checks only.


## Environment and kernel

The project is developed on Windows in the `csvi_env` conda environment, and the notebook metadata should preserve the `csvi_env` kernel. Package installation is outside the scope of this notebook. Missing optional dependencies for metadata inspection should be reported without stopping unrelated checks.


## Project root and portable paths

Notebook code should avoid absolute local paths. The setup cell discovers the project root by walking upward to marker files such as `README_FOR_CODEX.md`, `AGENTS.md`, or `pyproject.toml`, then defines canonical paths with `pathlib.Path`.


In [ ]:
import platform
import sys
from pathlib import Path

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root by walking upward from the current working directory."""
    current = Path.cwd() if start is None else Path(start).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    marker_list = ", ".join(MARKER_FILES)
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project folder "
        f"or make sure one of these marker files exists: {marker_list}."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
DOCS_DIR = PROJECT_ROOT / "docs"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
NOTEBOOKS_ACTIVE_DIR = PROJECT_ROOT / "notebooks_active"
NOTEBOOKS_ARCHIVE_DIR = PROJECT_ROOT / "notebooks_archive_READ_ONLY"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SRC_DIR = PROJECT_ROOT / "src"
THESIS_CONTEXT_DIR = PROJECT_ROOT / "thesis_context"

# Split-pipeline directories may not exist before implementation.
DATA_RAW_DIR = DATA_DIR / "raw"
DATA_EXTERNAL_DIR = DATA_DIR / "external"
DATA_INTERIM_DIR = DATA_DIR / "interim"
DATA_PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_TABLES_DIR = OUTPUTS_DIR / "tables"
OUTPUT_FIGURES_DIR = OUTPUTS_DIR / "figures"
OUTPUT_LOGS_DIR = OUTPUTS_DIR / "logs"

# Compatibility paths preserve the pre-split file layout.
CURRENT_DATASET_PATH = DATA_DIR / "Dataset.parquet"
CURRENT_WEATHER_OBS_PATH = DATA_DIR / "weather_final_master_all_vars.parquet"
CURRENT_MASTER_PANEL_PATH = DATA_DIR / "MASTERFILE_CALANDER.parquet"
CURRENT_FULL_CALENDAR_PATH = DATA_DIR / "df_catday_full_calendar.parquet"
CURRENT_ANALYSIS_READY_PATH = DATA_DIR / "df_final_analysis_ready.parquet"

# Planned split-notebook paths are declarations only.
MASTER_PANEL_PATH = DATA_PROCESSED_DIR / "master_sales_weather_calendar.parquet"
WEATHER_OBSERVED_DAILY_PATH = DATA_PROCESSED_DIR / "weather_observed_daily.parquet"
WEATHER_FORECAST_SCAFFOLD_PATH = DATA_PROCESSED_DIR / "weather_forecast_scaffold.parquet"
WEATHER_ENSEMBLE_FEATURES_PATH = DATA_PROCESSED_DIR / "weather_forecast_ensemble_features.parquet"
ML_FORECAST_PANEL_PATH = DATA_PROCESSED_DIR / "ml_forecast_panel.parquet"

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

In [ ]:
PATHS = {
    "project_root": PROJECT_ROOT,
    "data": DATA_DIR,
    "docs": DOCS_DIR,
    "notebooks": NOTEBOOKS_DIR,
    "notebooks_active": NOTEBOOKS_ACTIVE_DIR,
    "notebooks_archive_read_only": NOTEBOOKS_ARCHIVE_DIR,
    "outputs": OUTPUTS_DIR,
    "src": SRC_DIR,
    "thesis_context": THESIS_CONTEXT_DIR,
    "current_dataset": CURRENT_DATASET_PATH,
    "current_weather_observed": CURRENT_WEATHER_OBS_PATH,
    "current_master_panel": CURRENT_MASTER_PANEL_PATH,
    "future_master_panel": MASTER_PANEL_PATH,
    "future_weather_ensemble_features": WEATHER_ENSEMBLE_FEATURES_PATH,
    "future_ml_forecast_panel": ML_FORECAST_PANEL_PATH,
}

for name, path in PATHS.items():
    print(
        f"{name:34s} -> "
        f"{path.relative_to(PROJECT_ROOT) if path != PROJECT_ROOT else Path('.')}"
    )

## Source hierarchy

The project keeps legacy notebooks, thesis-context files, archive notebooks, and living documentation separate. The setup below records the current hierarchy so that later notebooks can refer to project files consistently and avoid editing read-only historical material.


## Active data files

The table records the main transaction, weather, calendar, and historical reference files used by the current pipeline checkpoint. Future split notebooks should write approved processed outputs under `data/processed/` rather than overwriting legacy files silently.

| File | Current role | Notes |
|---|---|---|
| `data/Dataset.parquet` | Active transaction-level source input | Used by `Hovedfilen.ipynb` to build categories, campaigns, basket analysis, and daily/category aggregates. |
| `data/weather_final_master_all_vars.parquet` | Active observed weather source input | Store-day realised weather used for the current master panel. |
| `data/df_final_analysis_ready.parquet` | Current intermediate | Written by the original notebook; not read later by the current notebook. |
| `data/df_catday_full_calendar.parquet` | Current intermediate | Full store-category-day calendar before final weather merge. |
| `data/MASTERFILE_CALANDER.parquet` | Current active processed panel | Full calendar with sales, campaign controls, closure flag, and observed weather. The filename is misspelled but should not be overwritten silently. |
| `data/JP_uStorOrdre.parquet` | Raw/source or raw reference, uncertain | Larger transaction export; not used by the current notebook. |
| `data/FINAL_MASTER_SALES_WEATHER.parquet` | Archive/stale processed output | Older item-level sales-weather merged table. |
| `data/FINAL_READY_FOR_MODEL.parquet` | Archive/stale or uncertain processed output | Older modelling-ready file. |


## Core identifiers

These identifiers define the sales date, forecast origin, horizon, store, region, category, weekday, and season fields used across the project. They are documented here to keep later data-construction and modelling notebooks aligned.

| Variable | Meaning |
|---|---|
| `DatoSolgt` | Original sales date. |
| `target_date` | Target sales/weather date for a forecast row. |
| `origin_date` | Forecast origin date. |
| `horizon` | Forecast horizon `h`, where `target_date = origin_date + h`. |
| `AvdelingID` | Store identifier. |
| `AvdelingTekst` | Store name. |
| `Region` | Region. |
| `Analyse_Kategori` | Analysis category. Current categories include `Dinner`, `Lunch`, `Sweets`, `Salads`, `Dessert`, `Cold Drinks`, `Hot Drinks`, and `Ekskluderes`. |
| `Ukedag` | Weekday. |
| `Årstid` | Season label used in realised-weather regression sections. |
| `season` | Lowercase English season label used in synthetic weather sections. |


## Sales variables

The sales variables distinguish the original-scale ML target, regression transformations, store-day totals, opening-status controls, and origin-safe historical sales features.

| Variable | Meaning |
|---|---|
| `Antall_total` | Store-category-day sales quantity. This is the main ML target on the original scale. |
| `log_Salg` | `log(1 + Antall_total)`. Used in regression because category sales can be zero even when the store is open. |
| `Avdeling_total` | Total daily sales quantity across categories for a store. |
| `Closed` | Closure indicator. Current logic marks a store-day closed when total sales are zero or when suspect microsales occur on Sundays/holidays. |
| `TotalSales` | Store-day total sales used in store-day regression variants. |
| `sales_lag_*` / `sales_roll*` | ML history features that must be computed using only information available at the forecast origin. |


## Campaign variables

Campaign variables are retained separately because same-day campaign volume can be endogenous to sales, while binary campaign indicators can be used as controls where appropriate.

| Variable | Meaning |
|---|---|
| `Is_Campaign` | Transaction/item-level campaign marker based on price group, discount, or selected product names. |
| `Antall_campaign` | Quantity sold through campaign-marked items. |
| `Antall_ordinary` | Quantity sold outside campaign-marked items. |
| `CampaignShare` | Share of store-category-day volume sold through campaign items. |
| `CampaignDay` | Indicator for any campaign activity in a store-category-day row. |
| `CampaignShare_total` | Store-day campaign share used in store-day regression variants. |
| `CampaignDay_total` | Store-day campaign indicator. |


## Observed weather variables

Observed weather variables are realised target-day measurements. They support descriptive analysis, realised-weather regressions, calibration diagnostics, and the M3 perfect-information benchmark, but they must not be used as operational ML forecast inputs.

| Variable | Meaning |
|---|---|
| `precip_val` / `precip_obs` | Observed precipitation. |
| `temp_mean` / `temp_obs` | Observed mean temperature. |
| `humidity_mean` / `humidity_obs` | Observed relative humidity. |
| `wind_mean` / `wind_obs` | Observed wind speed. |
| `cloud_mean` / `cloud_obs` | Observed cloud cover. |
| `apparent_temp` | Derived apparent temperature used in realised-weather regression analysis. |
| `dewpoint_obs` | Derived observed dew point. |
| `spread_obs` | Observed temperature minus dew point spread. |
| `AT_percentile`, `Precip_pct`, `Cloud_pct` | Store-season realised weather percentile features used in regression sections. |


## Synthetic weather variables

Synthetic weather variables represent forecast-origin-aligned weather inputs for operational ML models. They are forecast inputs rather than realised target-day measurements.

| Variable | Meaning |
|---|---|
| `temp_fcst` | Synthetic temperature forecast. |
| `wind_fcst` | Synthetic wind forecast. |
| `spread_fcst` | Synthetic temperature-dew point spread forecast. |
| `pop_fcst` | Probability of precipitation forecast. |
| `wet_fcst` | Simulated wet-day indicator. |
| `precip_fcst` | Simulated precipitation amount conditional on wet-day occurrence. |
| `cloud_open_prob`, `cloud_partly_prob`, `cloud_overcast_prob` | Forecast cloud regime probabilities. |
| `cloud_regime_fcst` | Simulated cloud regime. |
| `cloud_fcst` | Continuous cloud proxy generated from regime simulation. |
| `weather_forecast_ensemble_features` | Future ensemble feature table for the ML panel. |


## Forecast notation

The notation separates the forecast origin `t`, the target date `t+h`, and the horizon `h`. Forecast-weather inputs are denoted `W_hat_{i,t+h|t}`. Realised target-day weather is denoted `W_{i,t+h}` and is excluded from operational ML forecast inputs.


## Methodological guardrails

The project is a forecasting and decision-support study, not a causal identification design. ML weather inputs must represent information available at the forecast origin, and sales lags or rolling features must be computed only from information available at that origin. Regression uses `log(1 + Q_itk)` for store-category-day sales, while the main ML target remains original-scale `Q_itk` / `Antall_total`. Apparent temperature is used in realised-weather regression; forecasted apparent temperature is a robustness feature only. Synthetic-weather calibration values, season multipliers, scenario multipliers, and horizon definitions should not be changed without an explicit methodology decision.


## Lightweight data availability checks

The following check verifies expected file locations and, where possible, reads Parquet metadata only. It does not load full datasets.


In [ ]:
def format_bytes(num_bytes: int | None) -> str:
    if num_bytes is None:
        return "n/a"
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:,.1f} {unit}"
        size /= 1024


def parquet_metadata(path: Path) -> dict:
    """Return lightweight Parquet metadata without loading the full dataset."""
    if not path.exists():
        return {"rows": None, "columns": None, "column_names": []}
    try:
        import pyarrow.parquet as pq
        parquet_file = pq.ParquetFile(path)
        return {
            "rows": parquet_file.metadata.num_rows,
            "columns": len(parquet_file.schema_arrow.names),
            "column_names": parquet_file.schema_arrow.names,
        }
    except Exception as exc:
        return {
            "rows": None,
            "columns": None,
            "column_names": [],
            "schema_error": f"{type(exc).__name__}: {exc}",
        }


DATA_FILE_CHECKS = [
    {
        "name": "Dataset.parquet",
        "path": CURRENT_DATASET_PATH,
        "role": "required current transaction source",
        "required": True,
    },
    {
        "name": "weather_final_master_all_vars.parquet",
        "path": CURRENT_WEATHER_OBS_PATH,
        "role": "required current observed weather source",
        "required": True,
    },
    {
        "name": "MASTERFILE_CALANDER.parquet",
        "path": CURRENT_MASTER_PANEL_PATH,
        "role": "required current realised-weather master panel",
        "required": True,
    },
    {
        "name": "df_final_analysis_ready.parquet",
        "path": CURRENT_ANALYSIS_READY_PATH,
        "role": "current intermediate",
        "required": False,
    },
    {
        "name": "df_catday_full_calendar.parquet",
        "path": CURRENT_FULL_CALENDAR_PATH,
        "role": "current intermediate full calendar",
        "required": False,
    },
    {
        "name": "weather_forecast_ensemble_features.parquet",
        "path": WEATHER_ENSEMBLE_FEATURES_PATH,
        "role": "future processed forecast feature table",
        "required": False,
    },
]

rows = []
missing_required = []

for item in DATA_FILE_CHECKS:
    path = item["path"]
    exists = path.exists()
    if item["required"] and not exists:
        missing_required.append(path)
    stat_size = path.stat().st_size if exists else None
    meta = parquet_metadata(path)
    rows.append({
        "file": item["name"],
        "relative_path": (
            str(path.relative_to(PROJECT_ROOT))
            if path.is_relative_to(PROJECT_ROOT)
            else str(path)
        ),
        "role": item["role"],
        "required": item["required"],
        "exists": exists,
        "size": format_bytes(stat_size),
        "rows": meta.get("rows"),
        "columns": meta.get("columns"),
        "schema_error": meta.get("schema_error", ""),
        "column_names": meta.get("column_names", []),
    })

try:
    import pandas as pd
    display_columns = [
        "file", "relative_path", "role", "required", "exists",
        "size", "rows", "columns", "schema_error",
    ]
    display(pd.DataFrame(rows)[display_columns])
except Exception:
    for row in rows:
        print(row)

if missing_required:
    missing_lines = "\n".join(
        f"- {path.relative_to(PROJECT_ROOT)}" for path in missing_required
    )
    raise FileNotFoundError(
        "One or more required data files are missing. Place the files at these project-relative paths:\n"
        f"{missing_lines}"
    )

print("Required current data files are available.")

In [ ]:
# Metadata-only column inspection; full datasets are not loaded.
for row in rows:
    if not row["exists"] or not row["column_names"]:
        continue
    print(f"\n{row['file']} ({row['columns']} columns):")
    print(", ".join(row["column_names"]))

## Navigation

The main project references are `docs/PROJECT_MAP.md`, `docs/CODEX_RULES.md`, `docs/INVENTORY_AUDIT.md`, and `docs/REPLICATION_SPLIT_PLAN.md`. Later notebooks should be created, edited, or run only when that implementation step is explicitly requested.
